# DINO vs iBOT vs TDV on MELD Video Emotion Recognition

This Kaggle notebook extracts frozen video embeddings from the released SSv2-pretrained DINO, iBOT, and TDV encoders, then trains the same MLP classifier for seven-way MELD emotion recognition.

In [ ]:
# Cell 1: Install lightweight dependencies used for loading checkpoints and model utilities.
!pip install -q huggingface_hub einops timm seaborn


In [ ]:
# Cell 2: Import libraries, register the mounted TDV repo, and set global paths.
from pathlib import Path
import os, sys, gc, json, random, time
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from huggingface_hub import hf_hub_download, HfApi

INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
TDV_REPO = Path('/kaggle/input/datasets/pushkarsingh2005/tdv-code')

assert TDV_REPO.exists(), f'TDV repo not found: {TDV_REPO}'
sys.path.insert(0, str(TDV_REPO))

from model.model_utils import create_image_encoder, encode_images, get_cv_transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('TDV repo:', TDV_REPO)


In [ ]:
# Cell 3: Configure the benchmark. Use LIMIT_PER_SPLIT for a quick smoke test, then set it to None.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
LABEL_TO_ID = {x: i for i, x in enumerate(LABELS)}

MODEL_SIZE = 'small'       # 'small' first; later try 'base'
LIMIT_PER_SPLIT = None     # set to 50 or 100 for a smoke test
NUM_FRAMES = 8             # try 8 first; later try 16
FRAME_BATCH_SIZE = 32
IMAGE_SIZE = 224
POOLING = 'cls_mean'       # cls_mean, patch_mean, or cls_patch_concat

EMB_DIR = WORK_DIR / 'embeddings_tdv_dino_ibot'
OUT_DIR = WORK_DIR / 'benchmark_results_tdv_dino_ibot'
EMB_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL_SIZE:', MODEL_SIZE)
print('LIMIT_PER_SPLIT:', LIMIT_PER_SPLIT)
print('NUM_FRAMES:', NUM_FRAMES)


In [ ]:
# Cell 4: Auto-detect the mounted MELD root from Kaggle input folders.
SPLIT_LAYOUTS = {
    'train': {
        'csv': ['train/train_sent_emo.csv', 'train_sent_emo.csv'],
        'video_dirs': ['train/train_splits', 'train_splits'],
    },
    'dev': {
        'csv': ['dev_sent_emo.csv', 'dev/dev_sent_emo.csv'],
        'video_dirs': ['dev_splits_complete', 'dev/dev_splits_complete', 'dev_splits', 'dev/dev_splits'],
    },
    'test': {
        'csv': ['test_sent_emo.csv', 'test/test_sent_emo.csv'],
        'video_dirs': ['output_repeated_splits_test', 'test/output_repeated_splits_test', 'test_splits', 'test/test_splits'],
    },
}

def first_existing(root: Path, candidates):
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    return None

def looks_like_meld_root(root: Path):
    return (
        first_existing(root, SPLIT_LAYOUTS['train']['csv']) is not None
        and first_existing(root, SPLIT_LAYOUTS['train']['video_dirs']) is not None
    )

candidate_roots = []
for csv_path in INPUT_ROOT.rglob('train_sent_emo.csv'):
    candidate_roots.extend([csv_path.parent, csv_path.parent.parent])
candidate_roots = sorted(set(candidate_roots), key=lambda p: len(str(p)))

meld_root = None
for root in candidate_roots:
    if looks_like_meld_root(root):
        meld_root = root
        break

if meld_root is None:
    print('Could not auto-detect MELD root. Found CSVs:')
    for p in INPUT_ROOT.rglob('*sent_emo.csv'):
        print(p)
    raise FileNotFoundError('Set meld_root manually.')

print('MELD root:', meld_root)


In [ ]:
# Cell 5: Build clean train/dev/test DataFrames with verified video paths and integer emotion labels.
REQUIRED_COLUMNS = ['Utterance', 'Emotion', 'Dialogue_ID', 'Utterance_ID']

def clean_text(value):
    text = '' if pd.isna(value) else str(value)
    replacements = {'\u0092': "'", '\u0091': "'", '\u0093': '"', '\u0094': '"', '\u0096': '-', '\u0097': '-', '\u00a0': ' '}
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return ' '.join(text.split())

def prepare_split(meld_root: Path, split: str, limit=None):
    layout = SPLIT_LAYOUTS[split]
    csv_path = first_existing(meld_root, layout['csv'])
    video_dir = first_existing(meld_root, layout['video_dirs'])
    if csv_path is None:
        raise FileNotFoundError(f'No CSV for {split}')
    if video_dir is None:
        raise FileNotFoundError(f'No video dir for {split}')

    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f'{csv_path} missing columns: {missing}')

    out = pd.DataFrame()
    out['sample_id'] = df.apply(lambda r: f"{split}_dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}", axis=1)
    out['split'] = split
    out['dialogue_id'] = df['Dialogue_ID'].astype(int)
    out['utterance_id'] = df['Utterance_ID'].astype(int)
    out['utterance'] = df['Utterance'].map(clean_text)
    out['emotion'] = df['Emotion'].astype(str).str.lower().str.strip()
    out['emotion_id'] = out['emotion'].map(LABEL_TO_ID)
    out['video_file'] = out.apply(lambda r: f"dia{r['dialogue_id']}_utt{r['utterance_id']}.mp4", axis=1)
    out['video_path'] = out['video_file'].map(lambda name: str((video_dir / name).resolve()))
    out['video_exists'] = out['video_path'].map(lambda p: Path(p).exists())
    out = out[out['video_exists']].copy()
    out['emotion_id'] = out['emotion_id'].astype(int)
    out = out.reset_index(drop=True)
    if limit is not None:
        out = out.head(limit).copy()

    print(split, 'rows:', len(out), 'csv:', csv_path, 'video_dir:', video_dir)
    print(out['emotion'].value_counts().reindex(LABELS, fill_value=0).to_string())
    print()
    return out

indexes = {split: prepare_split(meld_root, split, LIMIT_PER_SPLIT) for split in ['train', 'dev', 'test']}
for split, df in indexes.items():
    df.to_csv(OUT_DIR / f'meld_{split}_index.csv', index=False)


In [ ]:
# Cell 6: Inspect the released Hugging Face checkpoint filenames so loading uses the real paths.
repo_id = 'ninaddaithankar/tdv'
files = HfApi().list_repo_files(repo_id)

print('Available checkpoint-like files:')
for f in files:
    if f.endswith(('.pth', '.ckpt', '.pt')):
        print(f)


In [ ]:
# Cell 7: Define checkpoint-loading helpers for TDV, DINO, and iBOT frame encoders.
def strip_prefix_if_present(sd, prefix):
    if any(k.startswith(prefix) for k in sd.keys()):
        return {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in sd.items()}
    return sd

def normalize_state_dict_keys(sd):
    for prefix in ['module.', 'model.', 'backbone.', 'encoder.', 'teacher.', 'student.']:
        sd = strip_prefix_if_present(sd, prefix)
    return sd

def get_checkpoint_filename(model_name, model_size):
    candidates = {
        ('tdv', 'small'): ['checkpoints/tdv-small.ckpt', 'checkpoints/tdv-small.pth', 'tdv-small.ckpt', 'tdv-small.pth'],
        ('tdv', 'base'):  ['checkpoints/tdv-base.ckpt',  'checkpoints/tdv-base.pth',  'tdv-base.ckpt',  'tdv-base.pth'],
        ('dino', 'small'): ['checkpoints/dino-small.pth', 'checkpoints/dino-small.ckpt', 'dino-small.pth', 'dino-small.ckpt'],
        ('dino', 'base'):  ['checkpoints/dino-base.pth',  'checkpoints/dino-base.ckpt',  'dino-base.pth',  'dino-base.ckpt'],
        ('ibot', 'small'): ['checkpoints/ibot-small.pth', 'checkpoints/ibot-small.ckpt', 'ibot-small.pth', 'ibot-small.ckpt'],
        ('ibot', 'base'):  ['checkpoints/ibot-base.pth',  'checkpoints/ibot-base.ckpt',  'ibot-base.pth',  'ibot-base.ckpt'],
    }[(model_name, model_size)]
    available = set(HfApi().list_repo_files(repo_id))
    for c in candidates:
        if c in available:
            return c
    raise FileNotFoundError(f'No checkpoint found for {model_name}-{model_size}. Tried: {candidates}')

def extract_encoder_state_from_checkpoint(ckpt, model_name):
    if model_name == 'tdv':
        sd = ckpt['state_dict']
        return {k.replace('model.frame_encoder.', ''): v for k, v in sd.items() if k.startswith('model.frame_encoder.')}
    if isinstance(ckpt, dict) and 'teacher' in ckpt:
        sd = ckpt['teacher']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError('Unknown checkpoint format')
    if not isinstance(sd, dict):
        raise ValueError('Selected state_dict is not a dict')
    return normalize_state_dict_keys(sd)

def try_load_encoder_state(model_name, model_size, enc_sd):
    patch_candidates = [14] if model_name == 'tdv' else [16, 14]
    last_error = None
    for patch_size in patch_candidates:
        encoder = create_image_encoder(backbone_type='dinov2', backbone_size=model_size, pretrained=False, patch_size=patch_size)
        try:
            encoder.load_state_dict(enc_sd, strict=True)
            print(f'Loaded {model_name}-{model_size} with patch_size={patch_size}, strict=True')
            return encoder, patch_size
        except Exception as e:
            last_error = e
        try:
            missing, unexpected = encoder.load_state_dict(enc_sd, strict=False)
            if len(unexpected) == 0 and len(missing) < 5:
                print(f'Loaded {model_name}-{model_size} with patch_size={patch_size}, strict=False')
                print('missing:', missing[:10], 'unexpected:', unexpected[:10])
                return encoder, patch_size
        except Exception as e:
            last_error = e
    raise RuntimeError(f'Could not load {model_name}-{model_size}. Last error: {last_error}')

def load_encoder(model_name, model_size):
    filename = get_checkpoint_filename(model_name, model_size)
    print(f'\nDownloading/loading {model_name}-{model_size}: {filename}')
    ckpt_path = hf_hub_download(repo_id=repo_id, filename=filename)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    enc_sd = extract_encoder_state_from_checkpoint(ckpt, model_name)
    print('encoder keys:', len(enc_sd))
    print('first key:', next(iter(enc_sd.keys())))
    encoder, patch_size = try_load_encoder_state(model_name, model_size, enc_sd)
    encoder = encoder.to(device).eval()
    for p in encoder.parameters():
        p.requires_grad_(False)
    return encoder, patch_size, filename


In [ ]:
# Cell 8: Define video frame sampling and frozen embedding extraction for one encoder.
def sample_video_frames(video_path, num_frames=8):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    indices = np.linspace(0, total - 1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame_bgr = cap.read()
        if ok:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
    cap.release()
    return frames

def embed_video(video_path, encoder, transform, num_frames=8, frame_batch_size=32):
    frames = sample_video_frames(video_path, num_frames=num_frames)
    if len(frames) == 0:
        raise RuntimeError(f'Could not decode frames: {video_path}')
    xs = torch.stack([transform(f) for f in frames])
    all_cls, all_patch_mean = [], []
    with torch.inference_mode():
        for start in range(0, xs.shape[0], frame_batch_size):
            batch = xs[start:start + frame_batch_size].to(device)
            tokens = encode_images(batch, 'dinov2', encoder)
            all_cls.append(tokens[:, 0].detach().cpu())
            all_patch_mean.append(tokens[:, 1:].mean(dim=1).detach().cpu())
    frame_cls = torch.cat(all_cls, dim=0)
    frame_patch_mean = torch.cat(all_patch_mean, dim=0)
    if POOLING == 'cls_mean':
        return frame_cls.mean(dim=0)
    if POOLING == 'patch_mean':
        return frame_patch_mean.mean(dim=0)
    if POOLING == 'cls_patch_concat':
        return torch.cat([frame_cls.mean(dim=0), frame_patch_mean.mean(dim=0)], dim=0)
    raise ValueError(f'Unknown pooling: {POOLING}')

def extract_split_embeddings(model_name, model_size, split, df, encoder, transform):
    out_path = EMB_DIR / f'meld_{split}_{model_name}_{model_size}_{POOLING}_{NUM_FRAMES}f.pt'
    if out_path.exists():
        print('Already exists:', out_path)
        return out_path
    embeddings, labels, sample_ids, errors = [], [], [], []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}-{model_size} {split}'):
        try:
            emb = embed_video(row.video_path, encoder, transform, NUM_FRAMES, FRAME_BATCH_SIZE)
            embeddings.append(emb)
            labels.append(int(row.emotion_id))
            sample_ids.append(row.sample_id)
        except Exception as e:
            errors.append({'sample_id': row.sample_id, 'video_path': row.video_path, 'error': repr(e)})
    if len(embeddings) == 0:
        raise RuntimeError(f'No embeddings extracted for {model_name}-{split}. First errors: {errors[:3]}')
    payload = {
        'embeddings': torch.stack(embeddings),
        'labels': torch.tensor(labels, dtype=torch.long),
        'sample_ids': sample_ids,
        'errors': errors,
        'config': {'model_name': model_name, 'model_size': model_size, 'pooling': POOLING, 'num_frames': NUM_FRAMES, 'image_size': IMAGE_SIZE},
    }
    torch.save(payload, out_path)
    print('Saved:', out_path)
    print('embeddings:', tuple(payload['embeddings'].shape), 'errors:', len(errors))
    return out_path

def extract_all_splits_for_model(model_name, model_size):
    encoder, patch_size, ckpt_file = load_encoder(model_name, model_size)
    transform, _ = get_cv_transforms(dataset_name='ssv2', image_dim=(IMAGE_SIZE, IMAGE_SIZE), custom_image_normalization=True)
    paths = {}
    for split in ['train', 'dev', 'test']:
        paths[split] = extract_split_embeddings(model_name, model_size, split, indexes[split], encoder, transform)
    del encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return paths


In [ ]:
# Cell 9: Define the MLP classifier and train/evaluate utilities used identically for every encoder.
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, num_classes=7, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))
    def forward(self, x):
        return self.net(x)

def load_payload(path):
    p = torch.load(path, map_location='cpu', weights_only=False)
    return p['embeddings'].float(), p['labels'].long(), p

def compute_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', labels=list(range(7)), zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', labels=list(range(7)), zero_division=0)),
    }

def evaluate_mlp(model, x, y, batch_size=256):
    model.eval()
    preds = []
    with torch.inference_mode():
        for start in range(0, x.shape[0], batch_size):
            logits = model(x[start:start + batch_size].to(device))
            preds.append(logits.argmax(dim=-1).cpu())
    y_pred = torch.cat(preds).numpy()
    y_true = y.numpy()
    return compute_metrics(y_true, y_pred), y_pred

def train_mlp_for_embeddings(model_name, model_size, paths, epochs=80, batch_size=128, lr=1e-3, weight_decay=1e-4):
    x_train, y_train, _ = load_payload(paths['train'])
    x_dev, y_dev, _ = load_payload(paths['dev'])
    x_test, y_test, _ = load_payload(paths['test'])

    scaler = StandardScaler()
    x_train = torch.tensor(scaler.fit_transform(x_train.numpy()), dtype=torch.float32)
    x_dev = torch.tensor(scaler.transform(x_dev.numpy()), dtype=torch.float32)
    x_test = torch.tensor(scaler.transform(x_test.numpy()), dtype=torch.float32)

    model = MLPClassifier(in_dim=x_train.shape[1]).to(device)
    counts = torch.bincount(y_train, minlength=7).float()
    class_weights = counts.sum() / (counts.clamp_min(1) * 7.0)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)

    best_dev_macro, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        dev_metrics, _ = evaluate_mlp(model, x_dev, y_dev)
        row = {'epoch': epoch, 'train_loss': float(np.mean(losses)), **{f'dev_{k}': v for k, v in dev_metrics.items()}}
        history.append(row)
        if dev_metrics['macro_f1'] > best_dev_macro:
            best_dev_macro = dev_metrics['macro_f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch == 1 or epoch % 10 == 0:
            print(f"{model_name}-{model_size} epoch {epoch:03d} loss={row['train_loss']:.4f} dev_acc={dev_metrics['accuracy']:.4f} dev_macro_f1={dev_metrics['macro_f1']:.4f}")

    model.load_state_dict(best_state)
    dev_metrics, dev_pred = evaluate_mlp(model, x_dev, y_dev)
    test_metrics, test_pred = evaluate_mlp(model, x_test, y_test)
    result = {
        'model_name': model_name, 'model_size': model_size, 'embedding_dim': int(x_train.shape[1]),
        'pooling': POOLING, 'num_frames': NUM_FRAMES, 'best_dev_macro_f1': float(best_dev_macro),
        'dev': dev_metrics, 'test': test_metrics,
        'dev_report': classification_report(y_dev.numpy(), dev_pred, target_names=LABELS, labels=list(range(7)), zero_division=0, output_dict=True),
        'test_report': classification_report(y_test.numpy(), test_pred, target_names=LABELS, labels=list(range(7)), zero_division=0, output_dict=True),
        'dev_confusion_matrix': confusion_matrix(y_dev.numpy(), dev_pred, labels=list(range(7))).tolist(),
        'test_confusion_matrix': confusion_matrix(y_test.numpy(), test_pred, labels=list(range(7))).tolist(),
        'history': history,
    }
    with open(OUT_DIR / f'result_{model_name}_{model_size}_{POOLING}_{NUM_FRAMES}f.json', 'w') as f:
        json.dump(result, f, indent=2)
    torch.save({'model_state_dict': model.state_dict(), 'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_, 'result': result}, OUT_DIR / f'mlp_{model_name}_{model_size}_{POOLING}_{NUM_FRAMES}f.pt')
    print('\nFINAL', model_name, model_size)
    print('dev:', dev_metrics)
    print('test:', test_metrics)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [ ]:
# Cell 10: Run the full DINO vs iBOT vs TDV benchmark with identical extraction and MLP settings.
MODELS_TO_RUN = [('dino', MODEL_SIZE), ('ibot', MODEL_SIZE), ('tdv', MODEL_SIZE)]
all_results = []

for model_name, model_size in MODELS_TO_RUN:
    print('\n' + '=' * 80)
    print('EXTRACTING:', model_name, model_size)
    print('=' * 80)
    paths = extract_all_splits_for_model(model_name, model_size)

    print('\n' + '=' * 80)
    print('TRAINING MLP:', model_name, model_size)
    print('=' * 80)
    result = train_mlp_for_embeddings(model_name, model_size, paths, epochs=80, batch_size=128, lr=1e-3, weight_decay=1e-4)
    all_results.append(result)


In [ ]:
# Cell 11: Build and save the main benchmark summary table.
rows = []
for r in all_results:
    rows.append({
        'model': f"{r['model_name']}-{r['model_size']}",
        'embedding_dim': r['embedding_dim'], 'pooling': r['pooling'], 'num_frames': r['num_frames'],
        'dev_accuracy': r['dev']['accuracy'], 'dev_macro_f1': r['dev']['macro_f1'], 'dev_weighted_f1': r['dev']['weighted_f1'],
        'test_accuracy': r['test']['accuracy'], 'test_macro_f1': r['test']['macro_f1'], 'test_weighted_f1': r['test']['weighted_f1'],
    })

results_df = pd.DataFrame(rows).sort_values('dev_macro_f1', ascending=False)
summary_path = OUT_DIR / f'summary_{MODEL_SIZE}_{POOLING}_{NUM_FRAMES}f.csv'
results_df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
results_df


In [ ]:
# Cell 12: Save per-class F1 scores and plot confusion matrices for each model.
import matplotlib.pyplot as plt
import seaborn as sns

per_class_rows = []
for r in all_results:
    report = r['test_report']
    for label in LABELS:
        per_class_rows.append({
            'model': f"{r['model_name']}-{r['model_size']}",
            'class': label,
            'precision': report[label]['precision'],
            'recall': report[label]['recall'],
            'f1': report[label]['f1-score'],
            'support': report[label]['support'],
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_path = OUT_DIR / f'test_per_class_{MODEL_SIZE}_{POOLING}_{NUM_FRAMES}f.csv'
per_class_df.to_csv(per_class_path, index=False)
print('Saved:', per_class_path)
display(per_class_df)

for r in all_results:
    cm = np.array(r['test_confusion_matrix'])
    name = f"{r['model_name']}-{r['model_size']}"
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
    plt.title(f'Test Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    fig_path = OUT_DIR / f'confusion_{name}_{POOLING}_{NUM_FRAMES}f.png'
    plt.savefig(fig_path, dpi=160)
    plt.show()
    print('Saved:', fig_path)
